In [73]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import os
tf.__version__

'2.0.0'

In [74]:
if tf.test.is_gpu_available():
    print(tf.test.gpu_device_name())
else:
    print("TF cannot find GPU")

/device:GPU:0


In [75]:
# same thing we had earlier -- this just loads the numpy arrays
mnist = tf.keras.datasets.mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# this is now different
train_data = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
test_data = tf.data.Dataset.from_tensor_slices((test_images, test_labels))

In [76]:
train_images = (train_images.astype(np.float32) / 255.).reshape((-1, 784))
test_images = (test_images.astype(np.float32) / 255.).reshape((-1, 784))

In [77]:
train_labels = train_labels.astype(np.int32)
test_labels = test_labels.astype(np.int32)

In [78]:
train_data = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_data = train_data.batch(128)

test_data = tf.data.Dataset.from_tensor_slices((test_images, test_labels))
test_data = test_data.batch(128)

In [79]:
tf.random.uniform(
    shape=[],
    minval=-0.1,
    maxval=0.1,
    dtype=tf.dtypes.float32,
    seed=None,
    name=None
)

<tf.Tensor: id=7737724, shape=(), dtype=float32, numpy=-0.0825083>

We want to have one additional layer, that is why we are going to add an additonal pair of weights and biases

In [80]:
# first change: set up log dir and file writer(s)
import time
logdir = os.path.join("logs", "linear" + str(time.time()))
train_writer = tf.summary.create_file_writer(os.path.join(logdir, "train"))
test_writer = tf.summary.create_file_writer(os.path.join(logdir, "test"))

In [81]:
train_steps = 10000
test_steps = 10000
learning_rate = 0.1

### weights and bias for the connection between the input and the hidden layer ###
W1 = tf.Variable(tf.random.uniform([784, 64]), dtype=tf.float32) # For the start im going with 64 neurons in the hidden layer
b1 = tf.Variable(tf.random.uniform([64]), dtype=tf.float32)
### weights and bias for the connection between the hidden and the output layer ###
W2 = tf.Variable(tf.random.uniform([64, 10]), dtype=tf.float32)
b2 = tf.Variable(tf.random.uniform([10]), dtype=tf.float32) #10 neurons in the output layer for the 10 classes

In [88]:
for step in range(train_steps):
    for img_batch, lbl_batch in train_data:
        with tf.GradientTape() as tape:
            ### input to hidden ### 
            hidden_out = tf.add(tf.matmul(img_batch, W1), b1) ### gonna change "tf.matmul(img_batch, W) + b" to tf.add(...) --> seems cleaner
            hidden_out = tf.nn.relu(hidden_out) #activation via relu
            ### hidden to output ### 
            logits = tf.add(tf.matmul(hidden_out, W2), b2) 
            xent = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
                logits=logits, labels=lbl_batch))

        grads = tape.gradient(xent, [W1, b1, W2, b2])
        W1.assign_sub(learning_rate * grads[0])
        b1.assign_sub(learning_rate * grads[1])
        W2.assign_sub(learning_rate * grads[2])
        b2.assign_sub(learning_rate * grads[3])

        # change #2: log this stuff every time step (rather wasteful)
        with train_writer.as_default():
            tf.summary.scalar("loss", xent, step=step)
            tf.summary.histogram("logits", logits, step=step)
            tf.summary.histogram("weights", W1, step=step)
            tf.summary.histogram("weights", W2, step=step)

            if not step % 100:
                preds = tf.argmax(logits, axis=1, output_type=tf.int32)
                acc = tf.reduce_mean(tf.cast(tf.equal(preds, lbl_batch),
                                     tf.float32))
                print("Training Loss: {} Accuracy: {}".format(xent, acc))

Training Loss: 0.970165491104126 Accuracy: 0.765625
Training Loss: 0.992667555809021 Accuracy: 0.6484375
Training Loss: 0.7807438969612122 Accuracy: 0.703125
Training Loss: 0.9753105640411377 Accuracy: 0.6796875
Training Loss: 1.0616874694824219 Accuracy: 0.578125
Training Loss: 0.8857206106185913 Accuracy: 0.65625
Training Loss: 1.2298061847686768 Accuracy: 0.6796875
Training Loss: 1.0917218923568726 Accuracy: 0.609375
Training Loss: 1.137550950050354 Accuracy: 0.671875
Training Loss: 1.0303484201431274 Accuracy: 0.546875
Training Loss: 1.170637607574463 Accuracy: 0.59375
Training Loss: 0.9829998016357422 Accuracy: 0.640625
Training Loss: 1.0038225650787354 Accuracy: 0.6796875
Training Loss: 0.8794228434562683 Accuracy: 0.7109375
Training Loss: 0.9715975522994995 Accuracy: 0.640625
Training Loss: 1.0076088905334473 Accuracy: 0.640625
Training Loss: 0.8532124757766724 Accuracy: 0.71875
Training Loss: 0.8942862749099731 Accuracy: 0.703125
Training Loss: 1.0095881223678589 Accuracy: 0.71

KeyboardInterrupt: 

In [70]:
for step in range(test_steps):
    for img_batch, lbl_batch in test_data:
        logits_1 = tf.nn.relu(tf.matmul(img_batch, W1) + b1) 
        test_preds = tf.argmax(tf.nn.softmax(tf.matmul(logits_1, W2) + b2), axis=1, output_type=tf.int32)
        test_acc = tf.reduce_mean(tf.cast(tf.equal(test_preds, test_labels), tf.float32))
        with test_writer.as_default():
            tf.summary.scalar("accuracy", test_acc, step=step)
            print("Test accuracy: {}\n".format(test_acc))

InvalidArgumentError: Incompatible shapes: [128] vs. [10000] [Op:Equal]

In [19]:
print("Loss: {} Accuracy: {}".format(xent, acc))

Loss: 0.9006915092468262 Accuracy: 0.7265625
